# Limpieza de Datos y EDA - Egresos Hospitalarios 2024

Dataset del MINSAL con los egresos hospitalarios de Chile 2024.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df_raw = pd.read_csv('EGRESOS_2024.csv', sep=';', encoding='latin1', low_memory=False)
print('filas:', df_raw.shape[0], '  columnas:', df_raw.shape[1])


## 1. Analisis descriptivo

Primero veo cuantas filas y columnas tiene, los tipos de datos y como se ve la variable principal DIAS_ESTADA.


In [ ]:
print('Filas:', df_raw.shape[0], '  Columnas:', df_raw.shape[1])
print()
print(df_raw.dtypes)


In [ ]:
df_raw.head(5)

In [ ]:
df_raw.describe(include='all').T

In [ ]:
# ver valores unicos de cada columna
for col in df_raw.columns:
    n = df_raw[col].nunique(dropna=False)
    muestra = df_raw[col].dropna().unique()[:4].tolist()
    print(col, '->', n, 'unicos |', muestra)


In [ ]:
# distribucion de columnas categoricas
for col in ['SEXO', 'GLOSA_PREVISION', 'CONDICION_EGRESO', 'GLOSA_REGION_RESIDENCIA']:
    print('\n---', col, '---')
    vc = df_raw[col].value_counts(dropna=False)
    pct = (vc / len(df_raw) * 100).round(2)
    print(pd.concat([vc, pct.rename('%')], axis=1).head(8))


In [ ]:
# histograma de DIAS_ESTADA
dias = df_raw['DIAS_ESTADA']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(dias.clip(upper=60), bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Dias de estadia (max 60)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribucion de DIAS_ESTADA')

axes[1].hist(dias.clip(upper=60), bins=50, color='steelblue', edgecolor='white', alpha=0.7)
for p, lbl in [(25,'P25'), (50,'P50'), (75,'P75'), (90,'P90'), (99,'P99')]:
    v = dias.quantile(p/100)
    axes[1].axvline(v, linestyle='--', label=f'{lbl}={v:.0f}')
axes[1].set_xlabel('Dias de estadia (max 60)')
axes[1].set_title('Percentiles de DIAS_ESTADA')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Estadisticas DIAS_ESTADA:')
print(dias.describe().round(2))


El dataset tiene 1.667.349 filas y 15 columnas. La variable DIAS_ESTADA tiene muchos valores chicos
(la mediana es 2 dias) pero hay algunos muy grandes. La distribucion es asimetrica hacia la derecha.

Las columnas de texto como SEXO y PREVISION tienen un valor '*' en algunos registros que hay que tratar.


## 2. Re-ajuste de tipos de variables

Hay columnas que quedaron como texto pero son categorias. Las voy a convertir a category.
DIAG2 tiene casi todo nulo asi que la elimino. ANO_EGRESO es siempre 2024 asi que tampoco sirve.


In [ ]:
df = df_raw.copy()

# eliminar columnas que no sirven
print('DIAG2 nulos:', df['DIAG2'].isna().mean() * 100, '%')
df.drop(columns=['DIAG2'], inplace=True)

print('ANO_EGRESO valores:', df['ANO_EGRESO'].unique())
df.drop(columns=['ANO_EGRESO'], inplace=True)

# tipos numericos
df['DIAS_ESTADA'] = df['DIAS_ESTADA'].astype('int32')
df['CONDICION_EGRESO'] = df['CONDICION_EGRESO'].astype('int8')

# SEXO: mapear los codigos a etiquetas
sexo_map = {'1': 'Hombre', '2': 'Mujer', '*': 'No identificado'}
df['SEXO'] = df['SEXO'].map(sexo_map).astype('category')

# PREVISION
prev_map = {
    '1':'FONASA', '2':'ISAPRE', '3':'CAPREDENA', '4':'DIPRECA',
    '5':'SISA', '96':'Particular', '99':'Desconocida', '*':'No identificado'
}
df['PREVISION'] = df['PREVISION'].map(prev_map).fillna('Otra').astype('category')

# GRUPO_EDAD como categoria ordenada
orden_edad = [
    'menor a 7 días', '7 A 27 DIAS', '28 DIAS A 2 MES', '2 MESES A MENOS DE 1 AÑO',
    '1 A 4 AÑOS', '5 A 9 AÑOS', '10 A 14 AÑOS', '15 A 19 AÑOS',
    '20 A 24 AÑOS', '25 A 29 AÑOS', '30 A 34 AÑOS', '35 A 39 AÑOS',
    '40 A 44 AÑOS', '45 A 49 AÑOS', '50 A 54 AÑOS', '55 A 59 AÑOS',
    '60 A 64 AÑOS', '65 A 69 AÑOS', '70 A 74 AÑOS', '75 A 79 AÑOS',
    '80 A 84 AÑOS', '85 A MAS'
]
presentes = [g for g in orden_edad if g in df['GRUPO_EDAD'].unique()]
df['GRUPO_EDAD'] = pd.Categorical(df['GRUPO_EDAD'], categories=presentes, ordered=True)
df['edad_ordinal'] = df['GRUPO_EDAD'].cat.codes

# DIAG1
df['DIAG1'] = df['DIAG1'].str.strip().str.upper().astype('category')

# el resto de columnas object a category
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')

# variable objetivo
df['estadia_prolongada'] = (df['DIAS_ESTADA'] >= 7).astype('int8')

print()
print('Tipos finales:')
print(df.dtypes)


In [ ]:
# verificar que no quedan columnas object
no_cat = [c for c in df.columns if str(df[c].dtype) == 'object']
print('Columnas object restantes:', no_cat if no_cat else 'ninguna')
print()
print('SEXO categorias:', df['SEXO'].cat.categories.tolist())
print('PREVISION categorias:', df['PREVISION'].cat.categories.tolist())
print('GRUPO_EDAD ordenado:', df['GRUPO_EDAD'].cat.ordered)
print('Dataset:', df.shape[0], 'filas x', df.shape[1], 'columnas')


Todas las columnas de texto pasaron a category. El '*' en SEXO y PREVISION lo mapeé a 'No identificado'
para no perder esos registros. GRUPO_EDAD quedo ordenado. DIAG2 y ANO_EGRESO los elimine porque no
aportan informacion util.


## 3. Datos ausentes

Despues de convertir los tipos veo cuantos nulos quedan y que hacer con ellos.


In [ ]:
# ver nulos por columna
nulos = pd.DataFrame({
    'nulos': df.isnull().sum(),
    'pct': (df.isnull().sum() / len(df) * 100).round(2)
})
con_nulos = nulos[nulos['nulos'] > 0].sort_values('nulos', ascending=False)
print('Columnas con nulos:')
print(con_nulos if not con_nulos.empty else 'No hay nulos')

if not con_nulos.empty:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(con_nulos.index, con_nulos['pct'], color='salmon')
    ax.set_xlabel('% de nulos')
    ax.set_title('Nulos por columna')
    plt.tight_layout()
    plt.show()


In [ ]:
# ver si los 'No identificado' de SEXO y PREVISION son los mismos registros
mask_sexo = df['SEXO'] == 'No identificado'
mask_prev = df['PREVISION'] == 'No identificado'

print('SEXO No identificado:', mask_sexo.sum())
print('PREVISION No identificado:', mask_prev.sum())
print('Ambos:', (mask_sexo & mask_prev).sum())
print()
print('Son el mismo bloque de registros. Se conservan como categoria valida.')


In [ ]:
# regiones especiales
for reg in ['Ignorada', 'Extranjero']:
    n = (df['GLOSA_REGION_RESIDENCIA'] == reg).sum()
    print(f'Region "{reg}": {n} registros')

print()
print('Se mantienen en el analisis general pero se excluyen en comparaciones regionales.')


In [ ]:
# resumen de lo que se hizo con los datos ausentes
print('RESUMEN:')
print('DIAG2: eliminada (90% nulos)')
print('ANO_EGRESO: eliminada (constante)')
print('SEXO/PREVISION con *: conservados como No identificado')
print('Region Ignorada/Extranjero: conservados')
print()
for c in ['DIAS_ESTADA','CONDICION_EGRESO','DIAG1','SEXO','PREVISION','GRUPO_EDAD']:
    print(f'  {c}: nulos = {df[c].isnull().sum()}')


Las variables principales (DIAS_ESTADA, DIAG1, SEXO, GRUPO_EDAD) no tienen nulos.
Los 28.382 registros con 'No identificado' son el mismo bloque en SEXO y PREVISION,
los dejo como categoria valida. No elimine ni impute ningun registro.


## 4. Outliers en DIAS_ESTADA

Hay estadias muy largas. Voy a ver si son errores o casos reales.


In [ ]:
print('Estadisticas DIAS_ESTADA:')
print(df['DIAS_ESTADA'].describe().round(2))


In [ ]:
# criterio IQR para detectar outliers
Q1 = df['DIAS_ESTADA'].quantile(0.25)
Q3 = df['DIAS_ESTADA'].quantile(0.75)
IQR = Q3 - Q1
limite_sup = Q3 + 1.5 * IQR
p99 = df['DIAS_ESTADA'].quantile(0.99)

n_outliers = (df['DIAS_ESTADA'] > limite_sup).sum()
n_p99 = (df['DIAS_ESTADA'] > p99).sum()

print(f'Q1={Q1:.0f}  Q3={Q3:.0f}  IQR={IQR:.0f}')
print(f'Limite superior: {limite_sup:.0f} dias')
print(f'Outliers IQR: {n_outliers} ({n_outliers/len(df)*100:.2f}%)')
print(f'Outliers P99 (>{p99:.0f} dias): {n_p99} ({n_p99/len(df)*100:.2f}%)')


In [ ]:
# como afectan los outliers a la media
media_full = df['DIAS_ESTADA'].mean()
media_sin = df.loc[df['DIAS_ESTADA'] <= p99, 'DIAS_ESTADA'].mean()
mediana = df['DIAS_ESTADA'].median()

print(f'Media con outliers:  {media_full:.2f} dias')
print(f'Media sin outliers:  {media_sin:.2f} dias')
print(f'Mediana:             {mediana:.0f} dias')


In [ ]:
# los 15 casos con estadia mas larga
cols = ['DIAS_ESTADA','GRUPO_EDAD','SEXO','DIAG1','GLOSA_REGION_RESIDENCIA','CONDICION_EGRESO']
print('Top 15 estadias mas largas:')
print(df[cols].nlargest(15, 'DIAS_ESTADA').to_string(index=False))


In [ ]:
# graficos de outliers
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# boxplot completo
axes[0].boxplot(df['DIAS_ESTADA'].values)
axes[0].set_title('Box plot - escala completa')
axes[0].set_ylabel('Dias de estadia')

# boxplot acotado a 60
axes[1].boxplot(df['DIAS_ESTADA'].clip(upper=60).values)
axes[1].set_title('Box plot - hasta 60 dias')
axes[1].set_ylabel('Dias de estadia')

# barras por rango
rangos = [0, 6, 30, 60, 90, 180, 99999]
etiquetas = ['1-6', '7-30', '31-60', '61-90', '91-180', '>180']
dias_np = df['DIAS_ESTADA'].values
conteos = [int(((dias_np > rangos[i]) & (dias_np <= rangos[i+1])).sum()) for i in range(len(etiquetas))]
axes[2].bar(etiquetas, conteos, color='steelblue')
axes[2].set_xlabel('Rango dias')
axes[2].set_ylabel('N de egresos')
axes[2].set_title('Distribucion por rango')

plt.tight_layout()
plt.show()


In [ ]:
# agrego una columna flag para identificar outliers extremos
df['outlier_extremo'] = (df['DIAS_ESTADA'] > 180).astype('int8')

print('Outliers extremos (>180 dias):', df['outlier_extremo'].sum(), f'({df["outlier_extremo"].mean()*100:.2f}%)')
print()
print('Decidi no eliminarlos porque los casos de estadias muy largas parecen')
print('ser hospitalizaciones por salud mental, demencias, etc. Son casos reales.')
print('Agrego el flag "outlier_extremo" por si se necesita filtrar despues.')


Segun el criterio IQR el 14% de los registros son outliers, pero eso es normal en datos hospitalarios
porque la distribucion es muy asimetrica. Los casos con estadia >180 dias corresponden principalmente
a diagnosticos de salud mental y adultos mayores, por lo que son clinicamente validos.

Decidi no eliminarlos. Agrego un flag y uso 'estadia_prolongada' (>=7 dias) como variable objetivo.


## 5. Relaciones entre variables

Veo como se relacionan las variables con la estadia prolongada.


In [ ]:
# correlacion de Spearman entre variables numericas
df['sexo_num'] = df['SEXO'].cat.codes
df['prev_num'] = df['PREVISION'].cat.codes
df['cond_num'] = df['CONDICION_EGRESO'].astype(float)

vars_num = ['edad_ordinal','sexo_num','prev_num','cond_num','DIAS_ESTADA','estadia_prolongada']
nombres = ['Edad','Sexo','Prevision','Cond. egreso','Dias estadia','Est. prolongada']

corr_sp = df[vars_num].corr(method='spearman')
corr_sp.columns = nombres
corr_sp.index = nombres

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_sp, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlacion Spearman')
plt.tight_layout()
plt.show()

print('Correlacion con estadia_prolongada:')
print(corr_sp['Est. prolongada'].drop('Est. prolongada').sort_values(ascending=False).round(4))


In [ ]:
# estadia por grupo de edad
edad_stats = (
    df.groupby('GRUPO_EDAD', observed=True)
    .agg(pct_prol=('estadia_prolongada','mean'), mediana=('DIAS_ESTADA','median'), n=('DIAS_ESTADA','count'))
    .reset_index()
)
edad_stats['pct_prol'] = edad_stats['pct_prol'] * 100

etiq = [str(g).replace(' AÑOS','').replace('menor a 7 días','<7d').replace('85 A MAS','85+') for g in edad_stats['GRUPO_EDAD']]

fig, ax1 = plt.subplots(figsize=(14, 4))
ax2 = ax1.twinx()
x = range(len(edad_stats))
ax1.bar(x, edad_stats['mediana'], color='steelblue', alpha=0.6, label='Mediana dias')
ax2.plot(x, edad_stats['pct_prol'], color='red', marker='o', lw=2, label='% prolongada')
ax1.set_xticks(list(x))
ax1.set_xticklabels(etiq, rotation=45, ha='right', fontsize=7)
ax1.set_ylabel('Mediana de dias')
ax2.set_ylabel('% estadia prolongada')
ax1.set_title('Estadia por Grupo de Edad')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(edad_stats[['GRUPO_EDAD','mediana','pct_prol','n']].to_string(index=False))


In [ ]:
# estadia por sexo
sexo_stats = (
    df.groupby('SEXO', observed=True)
    .agg(pct_prol=('estadia_prolongada','mean'), mediana=('DIAS_ESTADA','median'), n=('DIAS_ESTADA','count'))
    .reset_index()
)
sexo_stats['pct_prol'] = sexo_stats['pct_prol'] * 100
print(sexo_stats.round(2))

# test estadistico
h = df[df['SEXO']=='Hombre']['DIAS_ESTADA']
m = df[df['SEXO']=='Mujer']['DIAS_ESTADA']
u, p = stats.mannwhitneyu(h, m, alternative='two-sided')
print(f'\nMann-Whitney p={p:.4f}')
if p < 0.05:
    print('-> Diferencia significativa entre hombres y mujeres')
else:
    print('-> No hay diferencia significativa')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# violin
grupos = [df[df['SEXO']==s]['DIAS_ESTADA'].clip(upper=30).values for s in ['Hombre','Mujer','No identificado']]
axes[0].violinplot(grupos, positions=[1,2,3], showmedians=True)
axes[0].set_xticks([1,2,3])
axes[0].set_xticklabels(['Hombre','Mujer','No ident.'])
axes[0].set_ylabel('Dias (max 30)')
axes[0].set_title('Distribucion por Sexo')

# barras
axes[1].bar(sexo_stats['SEXO'].astype(str), sexo_stats['pct_prol'], color=['steelblue','salmon','gray'])
axes[1].set_ylabel('% estadia prolongada')
axes[1].set_title('% Prolongada por Sexo')

plt.tight_layout()
plt.show()


In [ ]:
# estadia por region
media_global = df['estadia_prolongada'].mean() * 100

reg_stats = (
    df[~df['GLOSA_REGION_RESIDENCIA'].isin(['Ignorada','Extranjero'])]
    .groupby('GLOSA_REGION_RESIDENCIA', observed=True)
    .agg(pct_prol=('estadia_prolongada','mean'), n=('estadia_prolongada','count'))
    .reset_index()
)
reg_stats['pct_prol'] = reg_stats['pct_prol'] * 100
reg_stats = reg_stats.sort_values('pct_prol')

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(reg_stats['GLOSA_REGION_RESIDENCIA'].astype(str), reg_stats['pct_prol'], color='steelblue')
ax.axvline(media_global, color='red', lw=1.5, ls='--', label=f'Promedio ({media_global:.1f}%)')
ax.set_xlabel('% estadia prolongada')
ax.set_title('% Estadia Prolongada por Region')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# estadia por capitulo CIE-10
cap_map = {
    'A':'Infec./Parasit.','B':'Infec./Parasit.','C':'Neoplasias','D':'Sangre/Inmun.',
    'E':'Endocrinas','F':'Mental','G':'Nervioso','H':'Ojo/Oido',
    'I':'Circulatorio','J':'Respiratorio','K':'Digestivo','L':'Piel',
    'M':'Musculoesqueletico','N':'Genitourinario','O':'Embarazo/Parto',
    'P':'Perinatal','Q':'Malformaciones','R':'Sintomas/Signos',
    'S':'Traumatismos','T':'Traumatismos','V':'Causas externas',
    'W':'Causas externas','X':'Causas externas','Y':'Causas externas','Z':'Factores salud'
}
df['capitulo_cie10'] = df['DIAG1'].astype(str).str[0].map(cap_map).fillna('Otro').astype('category')

cap_stats = (
    df.groupby('capitulo_cie10', observed=True)
    .agg(pct_prol=('estadia_prolongada','mean'), mediana=('DIAS_ESTADA','median'), n=('DIAS_ESTADA','count'))
    .reset_index()
)
cap_stats['pct_prol'] = cap_stats['pct_prol'] * 100
cap_stats = cap_stats[cap_stats['n'] >= 1000].sort_values('pct_prol')

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(cap_stats['capitulo_cie10'].astype(str), cap_stats['pct_prol'], color='steelblue')
ax.axvline(media_global, color='red', lw=1.5, ls='--', label=f'Promedio ({media_global:.1f}%)')
ax.set_xlabel('% estadia prolongada')
ax.set_title('% Estadia Prolongada por Capitulo CIE-10')
ax.legend()
plt.tight_layout()
plt.show()

print('Top 5 capitulos con mas estadias prolongadas:')
print(cap_stats.sort_values('pct_prol', ascending=False).head(5)[['capitulo_cie10','n','pct_prol','mediana']].to_string(index=False))


In [ ]:
# heatmap: edad x capitulo CIE-10
pivot = (
    df.groupby(['GRUPO_EDAD','capitulo_cie10'], observed=True)['estadia_prolongada']
    .mean().unstack() * 100
)
pivot = pivot.dropna(thresh=8, axis=1)

fig, ax = plt.subplots(figsize=(15, 8))
sns.heatmap(pivot, ax=ax, cmap='RdYlGn_r', annot=True, fmt='.0f', linewidths=0.3,
            annot_kws={'size': 7})
ax.set_title('% Estadia Prolongada: Edad x Capitulo CIE-10')
ax.set_xlabel('Capitulo CIE-10')
ax.set_ylabel('Grupo de edad')
plt.tight_layout()
plt.show()


In [ ]:
# estadia por prevision
prev_stats = (
    df.groupby('PREVISION', observed=True)
    .agg(pct_prol=('estadia_prolongada','mean'), mediana=('DIAS_ESTADA','median'), n=('DIAS_ESTADA','count'))
    .reset_index()
)
prev_stats['pct_prol'] = prev_stats['pct_prol'] * 100
prev_stats = prev_stats[prev_stats['n'] >= 100].sort_values('pct_prol', ascending=False)
print(prev_stats.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(prev_stats['PREVISION'].astype(str), prev_stats['pct_prol'], color='steelblue')
ax.axhline(media_global, color='red', lw=1.5, ls='--', label=f'Promedio ({media_global:.1f}%)')
ax.set_ylabel('% estadia prolongada')
ax.set_title('% Estadia Prolongada por Prevision')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# resumen comparativo
rango_edad = edad_stats['pct_prol'].max() - edad_stats['pct_prol'].min()
rango_reg  = reg_stats['pct_prol'].max()  - reg_stats['pct_prol'].min()
rango_cap  = cap_stats['pct_prol'].max()  - cap_stats['pct_prol'].min()
rango_prev = prev_stats['pct_prol'].max() - prev_stats['pct_prol'].min()

print('Variacion en % estadia prolongada por variable:')
print(f'  Grupo de edad:   {rango_edad:.1f} pp')
print(f'  Region:          {rango_reg:.1f} pp')
print(f'  Capitulo CIE-10: {rango_cap:.1f} pp')
print(f'  Prevision:       {rango_prev:.1f} pp')
print()
print('Correlacion Spearman con estadia_prolongada:')
print(corr_sp['Est. prolongada'].drop('Est. prolongada').sort_values(ascending=False).round(4))


La mayor correlacion Spearman con estadia_prolongada la tiene la edad ordinal, seguida de la condicion de egreso.

El capitulo CIE-10 es la variable con mayor rango de variacion entre grupos (mas de 40 puntos porcentuales).
El heatmap muestra que los adultos mayores con diagnosticos mentales o neurologicos tienen el mayor riesgo
de tener una estadia prolongada.

En conclusion, el diagnostico y la edad son los predictores mas importantes individualmente, pero
interactuan entre si segun el heatmap.
